In [ ]:
!pip install opencv-python dlib imutils mediapipe deepface

In [ ]:
import cv2
import dlib
import numpy as np
from imutils import face_utils
import mediapipe as mp
from deepface import DeepFace
from multiprocessing import Pool
import os

In [ ]:
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

In [ ]:
def analyze_head_movement(segment):
    head_movements = []

    for frame in segment:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)
        for face in faces:
            shape = predictor(gray, face)
            shape = face_utils.shape_to_np(shape)
            head_movements.append(shape[30])  # Nose tip as a reference point

    if len(head_movements) == 0:
        return [], []
    head_movements = np.array(head_movements)
    freq = np.fft.fftfreq(len(head_movements), d=1/30)  # Assuming 30 FPS
    amplitude = np.abs(np.fft.fft(head_movements[:, 1]))  # Vertical movements
    return freq, amplitude

In [ ]:
def analyze_smile_intensity(segment):
    smile_intensities = []

    for frame in segment:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)
        for face in faces:
            shape = predictor(gray, face)
            shape = face_utils.shape_to_np(shape)
            left_corner = shape[48]
            right_corner = shape[54]
            top_lip = shape[62]
            bottom_lip = shape[66]
            smile_intensity = np.linalg.norm(left_corner - right_corner) / np.linalg.norm(top_lip - bottom_lip)
            smile_intensities.append(smile_intensity)

    return smile_intensities

In [ ]:
def analyze_eye_closure(segment):
    eye_closures = []

    for frame in segment:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)
        for face in faces:
            shape = predictor(gray, face)
            shape = face_utils.shape_to_np(shape)
            left_eye = shape[36:42]
            right_eye = shape[42:48]
            left_eye_aspect_ratio = (np.linalg.norm(left_eye[1] - left_eye[5]) + np.linalg.norm(left_eye[2] - left_eye[4])) / (2 * np.linalg.norm(left_eye[0] - left_eye[3]))
            right_eye_aspect_ratio = (np.linalg.norm(right_eye[1] - right_eye[5]) + np.linalg.norm(right_eye[2] - right_eye[4])) / (2 * np.linalg.norm(right_eye[0] - right_eye[3]))
            eye_closures.append((left_eye_aspect_ratio, right_eye_aspect_ratio))

    return eye_closures

In [ ]:

def analyze_lip_corner_depression(segment):
    lip_corners = []

    for frame in segment:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)
        for face in faces:
            shape = predictor(gray, face)
            shape = face_utils.shape_to_np(shape)
            left_corner = shape[48]
            right_corner = shape[54]
            lip_corners.append((left_corner, right_corner))

    return lip_corners

In [ ]:
def analyze_body_posture(segment):
    postures = []

    for frame in segment:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            shoulder_left = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
            shoulder_right = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
            hip_left = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
            hip_right = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]
            posture = (shoulder_left, shoulder_right, hip_left, hip_right)
            postures.append(posture)

    return postures

In [ ]:
def analyze_micro_expressions(segment):
    micro_expressions = []

    for frame in segment:
        emotions = DeepFace.analyze(frame, actions=['emotion'])
        micro_expressions.append(emotions)

    return micro_expressions

In [ ]:
def analyze_movement_dynamics(segment):
    movement_dynamics = []

    prvs = cv2.cvtColor(segment[0], cv2.COLOR_BGR2GRAY)
    hsv = np.zeros_like(segment[0])
    hsv[..., 1] = 255

    for frame in segment[1:]:
        next_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prvs, next_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        movement_dynamics.append(np.mean(mag))
        prvs = next_frame

    return movement_dynamics

Define a function to process each video segment and extract the features

In [ ]:
def process_video_segment(args):
    video_path, start_time, end_time = args
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_MSEC, start_time * 1000)

    frames = []
    while cap.get(cv2.CAP_PROP_POS_MSEC) <= end_time * 1000:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)

    cap.release()

    head_movements = analyze_head_movement(frames)
    smile_intensity = analyze_smile_intensity(frames)
    eye_closure = analyze_eye_closure(frames)
    lip_corner_depression = analyze_lip_corner_depression(frames)
    body_posture = analyze_body_posture(frames)
    micro_expressions = analyze_micro_expressions(frames)
    movement_dynamics = analyze_movement_dynamics(frames)

    return {
        'head_movement': head_movements,
        'smile_intensity': smile_intensity,
        'eye_closure': eye_closure,
        'lip_corner_depression': lip_corner_depression,
        'body_posture': body_posture,
        'micro_expressions': micro_expressions,
        'movement_dynamics': movement_dynamics,
    }


In [ ]:
def process_videos_concurrently(video_paths, segment_length=180):  # Segment length in seconds
    pool = Pool()

    tasks = []
    for video_path in video_paths:
        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        duration = frame_count // fps

        for start_time in range(0, duration, segment_length):
            end_time = min(start_time + segment_length, duration)
            tasks.append((video_path, start_time, end_time))

        cap.release()

    results = pool.map(process_video_segment, tasks)
    pool.close()
    pool.join()

    return results

video_paths = ['/content/WIN_20240225_13_32_52_Pro.mp4']
features = process_videos_concurrently(video_paths)